In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
import os 
from utils import BinnedStimSpikeDataset, bin_spike_response, open_model_and_data
from viz import compute_avg_spikes_across_trials
import yaml
from metrics import *

n_neurons = 63
n_stim_channels = 42


In [6]:
print (os.listdir('results/hist_cnn_NOTS_output_3_4_26'))

['sweep_summary.csv', 'trial_0141_learning_rate=5.0e-04_history=1_kernel_sizes=5_conv_channels=256_fc_dims=256_seed0', 'trial_0140_learning_rate=5.0e-04_history=1_kernel_sizes=3_conv_channels=256_fc_dims=256_seed0', 'trial_0140_learning_rate=5.0e-04_history=1_kernel_sizes=3_conv_channels=256_fc_dims=256_seed2', 'trial_0140_learning_rate=5.0e-04_history=1_kernel_sizes=3_conv_channels=256_fc_dims=256_seed0.zip', 'trial_0141_learning_rate=5.0e-04_history=1_kernel_sizes=5_conv_channels=256_fc_dims=256_seed2', '3_4_sweep_results.csv', 'trial_0141_learning_rate=5.0e-04_history=1_kernel_sizes=5_conv_channels=256_fc_dims=256_seed1', 'trial_0140_learning_rate=5.0e-04_history=1_kernel_sizes=3_conv_channels=256_fc_dims=256_seed1']


In [2]:
SWEEP_DIR = 'results/hist_cnn_NOTS_output_3_4_26/trial_0140_learning_rate=5.0e-04_history=1_kernel_sizes=3_conv_channels=256_fc_dims=256_seed2/'
history_model, history_cfg, history_raw, test_loader = open_model_and_data(SWEEP_DIR)
nonhistory_model, nonhistory_cfg, nonhistory_raw, nonhistory_test_loader = open_model_and_data('results/base_cnn_NOTS_output_3_4_26/trial_0055_learning_rate=1.0e-04_kernel_sizes=5x3x3_conv_channels=64x128x256_fc_dims=1024_seed2/')


FileNotFoundError: [Errno 2] No such file or directory: 'results/hist_cnn_NOTS_output_3_4_26/trial_0140_learning_rate=5.0e-04_history=1_kernel_sizes=3_conv_channels=256_fc_dims=256_seed2/best_stim_spike_model.pt'

In [1]:
from viz import plot_oracle_trials_by_pattern
plot_oracle_trials_by_pattern(history_raw, history_cfg, n_neurons, n_stim_channels, title='History CNN')

NameError: name 'history_raw' is not defined

In [ ]:

from models import predict_autoregressive
from scipy.stats import pearsonr
import torch

COARSE_BIN_MS = 60
fine_bin_ms = 10  # 10 ms
factor = COARSE_BIN_MS // fine_bin_ms  # 6
# factor = 0  # Set to 0 to disable coarsening and use original fine bins
print(f"=== Correlation at {COARSE_BIN_MS}ms resolution per neuron ===")

# Wrap models as (model, cfg, device) tuples for metrics functions
hist_device = next(history_model.parameters()).device
nohist_device = next(nonhistory_model.parameters()).device
hist_tuple = (history_model, None, hist_device)
nohist_tuple = (nonhistory_model, None, nohist_device)

# ----- Compute all three teacher-forced / LOO -----
hist_corrs    = get_per_neuron_temporal_corr(hist_tuple, test_loader, coarse_factor=factor)
nohist_corrs  = get_per_neuron_temporal_corr(nohist_tuple, nonhistory_test_loader, coarse_factor=factor)
loo_corrs     = get_loo_temporal_corr(test_loader.dataset, coarse_factor=factor)   # LOO only depends on ground truth

print(f"History model   — mean r = {hist_corrs.mean():.4f}  (median {np.median(hist_corrs):.4f})")
print(f"No-history model — mean r = {nohist_corrs.mean():.4f}  (median {np.median(nohist_corrs):.4f})")
print(f"LOO average      — mean r = {loo_corrs.mean():.4f}  (median {np.median(loo_corrs):.4f})")
print(f"Neurons history > LOO:    {(hist_corrs > loo_corrs).sum()}/{len(loo_corrs)}")
print(f"Neurons no-history > LOO: {(nohist_corrs > loo_corrs).sum()}/{len(loo_corrs)}")

# ----- Autoregressive predictions -----
print(f"\n=== Computing autoregressive predictions... ===")
ar_pred, ar_true = predict_autoregressive(hist_tuple, test_loader, chunk_length=None, coarse_factor=factor)

# Per-neuron AR correlation
n_neurons_here = ar_pred.shape[1]
ar_corrs = np.zeros(n_neurons_here)
for n in range(n_neurons_here):
    p = ar_pred[:, n, :].ravel()
    t = ar_true[:, n, :].ravel()
    if p.std() > 1e-8 and t.std() > 1e-8:
        ar_corrs[n] = pearsonr(p, t)[0]
print(f"History AR model — mean r = {ar_corrs.mean():.4f}  (median {np.median(ar_corrs):.4f})")
print(f"Neurons AR > LOO:         {(ar_corrs > loo_corrs).sum()}/{len(loo_corrs)}")

# --- Fraction of Variance Explained (FVE) for all four predictors ---
hist_pred, hist_true     = collect_model_preds_and_targets(hist_tuple, test_loader, coarse_factor=factor)
nohist_pred, nohist_true = collect_model_preds_and_targets(nohist_tuple, nonhistory_test_loader, coarse_factor=factor)
loo_pred, loo_true       = collect_loo_preds_and_targets(test_loader.dataset, coarse_factor=factor, loo_flag=True)

hist_fve_neurons, hist_fve_mean       = fraction_variance_explained(hist_true, hist_pred, global_variance=True)
nohist_fve_neurons, nohist_fve_mean   = fraction_variance_explained(nohist_true, nohist_pred, global_variance=True)
loo_fve_neurons, loo_fve_mean         = fraction_variance_explained(loo_true, loo_pred, global_variance=True)
ar_fve_neurons, ar_fve_mean           = fraction_variance_explained(ar_true, ar_pred, global_variance=True)

print(f"\n=== Fraction of Variance Explained (FVE) per neuron at {COARSE_BIN_MS}ms resolution ===")
print(f"History model    — mean global FVE = {hist_fve_mean:.4f}  (median {np.median(hist_fve_neurons):.4f})")
print(f"History AR model — mean global FVE = {ar_fve_mean:.4f}  (median {np.median(ar_fve_neurons):.4f})")
print(f"No-history model — mean global FVE = {nohist_fve_mean:.4f}  (median {np.median(nohist_fve_neurons):.4f})")
print(f"LOO average      — mean global FVE = {loo_fve_mean:.4f}  (median {np.median(loo_fve_neurons):.4f})")
print(f"Neurons history > LOO:    {(hist_fve_neurons > loo_fve_neurons).sum()}/{len(loo_fve_neurons)}")
print(f"Neurons AR > LOO:         {(ar_fve_neurons > loo_fve_neurons).sum()}/{len(loo_fve_neurons)}")
print(f"Neurons no-history > LOO: {(nohist_fve_neurons > loo_fve_neurons).sum()}/{len(loo_fve_neurons)}")


In [ ]:

import matplotlib.pyplot as plt

# Pre-computed per-neuron arrays for each "model" variant
MODEL_DATA = {
    'hist_gt': {
        'corrs': hist_corrs,
        'fve': hist_fve_neurons,
        'label': 'History (ground-truth)',
    },
    'hist_ar': {
        'corrs': ar_corrs,
        'fve': ar_fve_neurons,
        'label': 'History (autoregressive)',
    },
    'nohist': {
        'corrs': nohist_corrs,
        'fve': nohist_fve_neurons,
        'label': 'No-history',
    },
}

def plot_model_comparison(model_keys,
                          loo_corrs, loo_fve,
                          coarse_bin_ms,
                          save_path=None):
    """Scatter-plot one or more models against the LOO baseline (correlation & FVE).

    Parameters
    ----------
    model_keys : list of str
        Keys into MODEL_DATA, e.g. ['hist_gt', 'hist_ar', 'nohist'].
    loo_corrs : np.ndarray   – per-neuron LOO temporal correlations.
    loo_fve   : np.ndarray   – per-neuron LOO fraction variance explained.
    coarse_bin_ms : int       – bin width for axis title.
    save_path : str or None   – if given, save figure to this path.

    Returns
    -------
    fig, ax
    """
    n_total = len(loo_corrs)

    fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))

    for key in model_keys:
        d = MODEL_DATA[key]
        n_better_corr = (d['corrs'] > loo_corrs).sum()
        n_better_fve  = (d['fve']   > loo_fve).sum()

        ax[0].scatter(loo_corrs, d['corrs'], alpha=0.7, s=25,
                      label=f"{d['label']} ({n_better_corr}/{n_total} > LOO)")
        ax[1].scatter(loo_fve, d['fve'], alpha=0.7, s=25,
                      label=f"{d['label']} ({n_better_fve}/{n_total} > LOO)")

    for a in ax:
        a.plot([0, 1], [0, 1], 'k--', lw=0.8)
        a.legend(fontsize=8)

    ax[0].set_xlabel("LOO Temporal Correlation")
    ax[0].set_ylabel("Model Temporal Correlation")
    ax[0].set_title(f"Temporal Correlation per Neuron (coarse {coarse_bin_ms}ms bins)")

    ax[1].set_xlabel("LOO Fraction Variance Explained")
    ax[1].set_ylabel("Model Fraction Variance Explained")
    ax[1].set_title(f"Fraction Variance Explained per Neuron (coarse {coarse_bin_ms}ms bins)")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig, ax

# --- Compare all three: no-history, history GT, history AR ---
fig, ax = plot_model_comparison(
    ['nohist', 'hist_gt', 'hist_ar'],
    loo_corrs, loo_fve_neurons,
    COARSE_BIN_MS,
    save_path=os.path.join(SWEEP_DIR, f'{COARSE_BIN_MS}_nohist_vs_histGT_vs_histAR_scatter.png'),
)


In [ ]:

from viz import plot_single_oracle_trial

# --- Example usage ---
for pattern in [4001, 4035, 4042, 4048, 4036, 4047, 4026]:
    for trial in range(5):
        fig = plot_single_oracle_trial(
            pattern, trial, hist_tuple, history_cfg, history_raw, test_loader,
            coarse_factor=1, coarse_method='mean',
            nonhistory_model_tuple=nohist_tuple,
            nonhistory_cfg=nonhistory_cfg,
            nonhistory_test_loader=nonhistory_test_loader,
            save_path=os.path.join(SWEEP_DIR, f'oracle_trial_{pattern}_trial{trial}_coarse{COARSE_BIN_MS}ms.png'),
        )


In [ ]:
from models import perturbation_analysis
import matplotlib.pyplot as plt

N_SHUFFLES = 20

def plot_perturbation_results(results, coarse_bin_ms, n_shuffles, save_prefix=None, model_label=''):
    """Bar plot + per-neuron scatter for perturbation_analysis output.
    
    Automatically adapts to whether history perturbations are present.
    """
    has_history = 'history_shuffle' in results

    # --- Summary table ---
    suffix = f' ({model_label})' if model_label else ''
    print(f"=== Perturbation Analysis{suffix} ({n_shuffles} shuffles, {coarse_bin_ms}ms bins) ===\n")
    print(f"{'Condition':<25s}  {'Mean FEV':>10s}  {'Std':>8s}")
    print("-" * 50)
    print(f"{'Original':<25s}  {results['original']['fve_mean']:>10.4f}")
    pert_keys = ['stim_shuffle']
    if has_history:
        pert_keys += ['history_mean', 'history_shuffle']
    for key in pert_keys:
        r = results[key]
        if isinstance(r['fve_mean'], np.ndarray):
            m, s = r['fve_mean'].mean(), r['fve_mean'].std()
            print(f"{key:<25s}  {m:>10.4f}  {s:>8.4f}")
        else:
            print(f"{key:<25s}  {r['fve_mean']:>10.4f}")

    # --- Bar plot ---
    all_keys = ['original', 'stim_shuffle']
    labels = ['Original', 'Stim\nShuffle']
    colors = ['tab:blue', 'tab:orange']
    if has_history:
        all_keys += ['history_mean', 'history_shuffle']
        labels += ['History\nMean', 'History\nShuffle']
        colors += ['tab:red', 'tab:purple']

    means, stds = [], []
    for key in all_keys:
        r = results[key]
        if isinstance(r['fve_mean'], np.ndarray):
            means.append(r['fve_mean'].mean())
            stds.append(r['fve_mean'].std())
        else:
            means.append(r['fve_mean'])
            stds.append(0)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors, alpha=0.8)
    ax.set_ylabel('Fraction of Variance Explained over Test Dataset')
    title = f'Perturbation Analysis ({coarse_bin_ms}ms bins, {n_shuffles} shuffles)'
    if model_label:
        title += f'\n{model_label}'
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

    for bar, m, s in zip(bars, means, stds):
        txt = f'{m:.3f}' if s == 0 else f'{m:.3f}\u00b1{s:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.005,
                txt, ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    if save_prefix:
        plt.savefig(f'{save_prefix}_bar.png', dpi=300, bbox_inches='tight')
    plt.show()

    # --- Per-neuron scatter ---
    scatter_keys = [('stim_shuffle', 'Stim Shuffle')]
    if has_history:
        scatter_keys += [('history_mean', 'History Mean'),
                         ('history_shuffle', 'History Shuffle')]

    fig2, axes = plt.subplots(1, len(scatter_keys), figsize=(5*len(scatter_keys), 4.5))
    if len(scatter_keys) == 1:
        axes = [axes]
    for ax_i, (key, title) in zip(axes, scatter_keys):
        r = results[key]
        pert_fve = r['fve_neurons'].mean(axis=0) if r['fve_neurons'].ndim == 2 else r['fve_neurons']
        ax_i.scatter(results['original']['fve_neurons'], pert_fve, alpha=0.6, s=20)
        lims = [min(ax_i.get_xlim()[0], ax_i.get_ylim()[0]),
                max(ax_i.get_xlim()[1], ax_i.get_ylim()[1])]
        ax_i.plot(lims, lims, 'k--', lw=0.8)
        ax_i.set_xlabel('Original FEV over Test Dataset')
        ax_i.set_ylabel(f'{title} FEV')
        ax_i.set_title(title)
        ax_i.set_aspect('equal')

    suptitle = f'Per-Neuron FEV: Original vs Perturbation ({coarse_bin_ms}ms bins)'
    if model_label:
        suptitle += f' — {model_label}'
    fig2.suptitle(suptitle, y=1.02)
    plt.tight_layout()
    if save_prefix:
        plt.savefig(f'{save_prefix}_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()

    return fig, fig2

# --- History model perturbation ---
results_hist = perturbation_analysis(
    hist_tuple, test_loader,
    n_stim_channels=n_stim_channels,
    n_shuffles=N_SHUFFLES,
    coarse_factor=factor,
    seed=42,
)
plot_perturbation_results(
    results_hist, COARSE_BIN_MS, N_SHUFFLES,
    save_prefix=os.path.join(SWEEP_DIR, f'perturbation_history_{COARSE_BIN_MS}ms'),
    model_label='History Model',
)

In [ ]:
# --- No-history model perturbation (stim shuffle only) ---
results_nohist = perturbation_analysis(
    nohist_tuple, nonhistory_test_loader,
    n_stim_channels=n_stim_channels,
    n_shuffles=N_SHUFFLES,
    coarse_factor=factor,
    seed=42,
)
plot_perturbation_results(
    results_nohist, COARSE_BIN_MS, N_SHUFFLES,
    save_prefix=os.path.join(SWEEP_DIR, f'perturbation_nohist_{COARSE_BIN_MS}ms'),
    model_label='No-History Model',
)

In [ ]:
# --- Load best model per history value, collect predictions/targets ---
import pandas as pd
import os

COARSE_FACTOR = 1       # 60ms bins; change to 0 for fine (10ms) bins

# 1. Find best trial per history from sweep CSV
sweep_df = pd.read_csv(os.path.join(SWEEP_DIR, 'sweep_results.csv'))
best_idx = sweep_df.groupby('history')['test_corr'].idxmax()
best_df = sweep_df.loc[best_idx].sort_values('history').reset_index(drop=True)

print(f"Loading {len(best_df)} models (coarse_factor={COARSE_FACTOR})...")

# 2. For each history value, load model & store predictions/targets
history_values = []
model_kernel_sizes = []
sweep_preds = []    # list of (pred, true) arrays per history
sweep_loaders = []  # keep loaders for correlation

for _, row in best_df.iterrows():
    run_dir = row['run_dir']
    h = int(row['history'])
    history_values.append(h)
    
    model, cfg, raw_data, loader = open_model_and_data(run_dir)
    # Extract kernel size (assuming single kernel size in list like [60])
    ks = cfg.get('kernel_sizes', [60])
    model_kernel_sizes.append(str(ks[0]) if isinstance(ks, list) and len(ks) > 0 else str(ks))
    
    dev = next(model.parameters()).device
    pred, true = collect_model_preds_and_targets((model, None, dev), loader, coarse_factor=COARSE_FACTOR)
    sweep_preds.append((pred, true))
    sweep_loaders.append(loader)
    print(f"  history={h:>2d}  ks= {model_kernel_sizes[-1]}  run={run_dir.split('/')[-1]}  shape={pred.shape}")

print("Done. Use sweep_preds / sweep_loaders for FEV and correlation plots.")


In [ ]:
# --- FEV vs History ---
import matplotlib.pyplot as plt

# Dynamically compute baselines at sweep's COARSE_FACTOR
_loo_pred, _loo_true = collect_loo_preds_and_targets(sweep_loaders[0].dataset, coarse_factor=COARSE_FACTOR, loo_flag=True)
_, LOO_FVE_MEAN = fraction_variance_explained(_loo_true, _loo_pred, global_variance=True)

_nohist_dev = next(nonhistory_model.parameters()).device
_nh_pred, _nh_true = collect_model_preds_and_targets((nonhistory_model, None, _nohist_dev), nonhistory_test_loader, coarse_factor=COARSE_FACTOR)
_, nohist_fve_mean = fraction_variance_explained(_nh_true, _nh_pred, global_variance=True)

fve_means = []
for pred, true in sweep_preds:
    _, fve_mean = fraction_variance_explained(true, pred, global_variance=True)
    fve_means.append(fve_mean)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(history_values, fve_means, 'o-', color='tab:blue', label='History CNN (best per h)')

# Annotate with Kernel Size
for i, h in enumerate(history_values):
    ks = model_kernel_sizes[i]
    ax.text(h, fve_means[i] + 0.005, f'k={ks}', ha='center', va='bottom', fontsize=8, color='tab:blue')

ax.axhline(nohist_fve_mean, color='tab:orange', ls='--', lw=1.5, label=f'No-history CNN ({nohist_fve_mean:.3f})')
ax.axhline(LOO_FVE_MEAN, color='tab:green', ls=':', lw=1.5, label=f'LOO baseline ({LOO_FVE_MEAN:.3f})')
ax.set_xlabel('History (bins)')
ax.set_ylabel('Fraction of Explained Variance')

coarse_bin_ms = fine_bin_ms * COARSE_FACTOR if COARSE_FACTOR > 0 else fine_bin_ms
ax.set_title(f'FEV vs History Length ({coarse_bin_ms}ms bins, global variance)')
ax.set_xticks(history_values)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SWEEP_DIR, f'FEV_vs_history_coarse{coarse_bin_ms}ms.png'), dpi=300, bbox_inches='tight', transparent=False)
plt.show()


In [ ]:
%matplotlib inline
# --- Temporal Correlation vs History ---

# Dynamically compute baselines at sweep's COARSE_FACTOR
loo_corrs = get_loo_temporal_corr(sweep_loaders[0].dataset, coarse_factor=COARSE_FACTOR)
loo_corr_mean = loo_corrs.mean()

_nohist_dev = next(nonhistory_model.parameters()).device
nohist_corrs = get_per_neuron_temporal_corr((nonhistory_model, None, _nohist_dev), nonhistory_test_loader, coarse_factor=COARSE_FACTOR)
nohist_corr_mean = nohist_corrs.mean()

corr_means = []
for pred, true in sweep_preds:
    n_neurons_here = pred.shape[1]
    neuron_corrs = np.zeros(n_neurons_here)
    for n in range(n_neurons_here):
        p = pred[:, n, :].ravel()
        t = true[:, n, :].ravel()
        if p.std() > 1e-8 and t.std() > 1e-8:
            neuron_corrs[n] = pearsonr(p, t)[0]
    corr_means.append(neuron_corrs.mean())

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(history_values, corr_means, 'o-', color='tab:blue', label='History CNN (best per h)')
ax.axhline(nohist_corr_mean, color='tab:orange', ls='--', lw=1.5, label=f'No-history CNN ({nohist_corr_mean:.3f})')
ax.axhline(loo_corr_mean, color='tab:green', ls=':', lw=1.5, label=f'LOO baseline ({loo_corr_mean:.3f})')
ax.set_xlabel('History (bins)')
ax.set_ylabel('Mean Temporal Correlation (Pearson r)')
coarse_bin_ms = fine_bin_ms * COARSE_FACTOR if COARSE_FACTOR > 0 else fine_bin_ms
ax.set_title(f'Correlation vs History Length ({coarse_bin_ms}ms bins)')
ax.set_xticks(history_values)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
f = test_loader.dataset.visualize_sample(45, cfg=init_cfg)
plt.show()

In [ ]:

from viz import plot_neuron_pattern_traces
# # --- Single example ---
fig = plot_neuron_pattern_traces(0, 4041, hist_tuple, history_cfg, history_raw, test_loader,
                                 coarse_factor=6, save_dir=os.path.join(SWEEP_DIR, "sample_traces"))

# --- Generate all (uncomment to run; saves ~3150 figures) ---
# generate_all_neuron_pattern_plots(history_model, history_cfg, history_raw, test_loader,
#                                    coarse_factor=1,
#                                    save_dir='results/neuron_pattern_traces_coarse1')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from models import HistoryCacheCausalCNN, train_epoch, validate
from utils import BinnedStimSpikeDataset
from datetime import datetime

# ---- Config for cache model ----
CACHE_SIZE = 5
CACHE_EMBED_DIM = 16
CONV_CHANNELS = [256]
KERNEL_SIZES = [5]
FC_DIMS = [256]
DROPOUT = 0.2
N_EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 16

# ---- Training modes to iterate over ----
TRAIN_MODES = ['teacher_forcing', 'semi_ar', 'ar']

# ---- Use history_raw (already loaded in cell 2) ----
raw = history_raw
pattern_df = raw["pattern_df"]
unique_trials_info = pattern_df[["pattern_timing_index", "pattern_name", "is_oracle"]].drop_duplicates()
oracle_timing = unique_trials_info[unique_trials_info["is_oracle"]]["pattern_timing_index"].tolist()
sample_timing = unique_trials_info[~unique_trials_info["is_oracle"]]["pattern_timing_index"].tolist()
train_indices, val_indices = train_test_split(sample_timing, test_size=0.15, random_state=42)
test_indices = oracle_timing

# ---- Build datasets with history=0 (stim only) ----
ds_kwargs = dict(
    pattern_df=raw["pattern_df"],
    spike_responses=raw["spike_responses"],
    channel_to_index=raw["channel_to_index"],
    timing_to_pattern=raw["timing_to_pattern"],
    input_bin_size_ms=10,
    output_bin_size_ms=10,
    n_input_bins=60,
    n_output_bins=60,
    max_time_ms=600,
    output_offset=0,
    encoding_mode='current',
    init_state=False,
    n_initial_state_bins=0,
    history=0,
    logger=None,
)

train_ds = BinnedStimSpikeDataset(trial_indices=train_indices, **ds_kwargs)
val_ds   = BinnedStimSpikeDataset(trial_indices=val_indices,   **ds_kwargs)
test_ds  = BinnedStimSpikeDataset(trial_indices=test_indices,  **ds_kwargs)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
test_loader_cache = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

print(f"Train: {len(train_ds)} samples, Val: {len(val_ds)}, Test: {len(test_ds)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'

# ---- Train and save a model for each mode ----
trained_models = {}
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for TRAIN_MODE in TRAIN_MODES:
    print(f"\n{'='*60}")
    print(f"Training mode: {TRAIN_MODE}")
    print('='*60)
    
    # Create save directory
    save_dir = f'results/cache_cnn_{TRAIN_MODE}_{timestamp}'
    os.makedirs(save_dir, exist_ok=True)
    
    # Instantiate fresh model
    cache_model = HistoryCacheCausalCNN(
        n_stim_channels=n_stim_channels,
        n_neurons=n_neurons,
        n_input_bins=60,
        n_output_bins=60,
        conv_channels=CONV_CHANNELS,
        kernel_sizes=KERNEL_SIZES,
        fc_dims=FC_DIMS,
        dropout=DROPOUT,
        cache_size=CACHE_SIZE,
        cache_embed_dim=CACHE_EMBED_DIM,
        use_batch_norm=True,
        use_init_state=False,
    ).to(device)
    
    n_params = sum(p.numel() for p in cache_model.parameters())
    print(f"HistoryCacheCausalCNN: {n_params:,} parameters")
    
    # Training loop with AMP
    criterion = nn.PoissonNLLLoss(log_input=True, reduction='mean')
    optimizer = torch.optim.Adam(cache_model.parameters(), lr=LR, weight_decay=1e-5)
    scaler = GradScaler(enabled=USE_AMP)
    
    best_val_loss = float('inf')
    train_losses, val_losses = [], []
    
    for epoch in range(N_EPOCHS):
        cache_model.train()
        total_loss = 0
        n_batches = 0
        for bx, by in train_loader:
            bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
            optimizer.zero_grad()
            spike_ctx = None if TRAIN_MODE == 'ar' else by
            with autocast(enabled=USE_AMP):
                pred = cache_model(bx, spike_ctx, mode=TRAIN_MODE)
                loss = criterion(pred, by)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(cache_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            n_batches += 1
        train_loss = total_loss / n_batches
        train_losses.append(train_loss)
        
        # Validation
        cache_model.eval()
        val_loss = 0
        val_batches = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
                spike_ctx = None if TRAIN_MODE == 'ar' else by
                with autocast(enabled=USE_AMP):
                    pred = cache_model(bx, spike_ctx, mode=TRAIN_MODE)
                    vloss = criterion(pred, by)
                val_loss += vloss.item()
                val_batches += 1
        val_loss /= val_batches
        val_losses.append(val_loss)
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(cache_model.state_dict(), os.path.join(save_dir, 'best_model.pt'))
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{N_EPOCHS} — train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}")
    
    # Save final model and config
    torch.save(cache_model.state_dict(), os.path.join(save_dir, 'final_model.pt'))
    
    config = {
        'train_mode': TRAIN_MODE,
        'cache_size': CACHE_SIZE,
        'cache_embed_dim': CACHE_EMBED_DIM,
        'conv_channels': CONV_CHANNELS,
        'kernel_sizes': KERNEL_SIZES,
        'fc_dims': FC_DIMS,
        'dropout': DROPOUT,
        'n_epochs': N_EPOCHS,
        'lr': LR,
        'batch_size': BATCH_SIZE,
        'best_val_loss': best_val_loss,
        'final_train_loss': train_losses[-1],
        'final_val_loss': val_losses[-1],
    }
    with open(os.path.join(save_dir, 'config.yaml'), 'w') as f:
        yaml.dump(config, f)
    
    # Save loss curves
    np.savez(os.path.join(save_dir, 'losses.npz'), train=train_losses, val=val_losses)
    
    trained_models[TRAIN_MODE] = {
        'model': cache_model,
        'save_dir': save_dir,
        'best_val_loss': best_val_loss,
    }
    
    print(f"Saved to {save_dir} (best val_loss: {best_val_loss:.4f})")

print(f"\n{'='*60}")
print("All models trained and saved!")
for mode, info in trained_models.items():
    print(f"  {mode}: {info['save_dir']} (best val_loss: {info['best_val_loss']:.4f})")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from metrics import _coarsen
from torch.cuda.amp import autocast

# Use the same coarse bin size as cell 3 so LOO / hist / nohist comparisons are consistent
COARSE_BIN_MS = 60
coarse_factor = COARSE_BIN_MS // fine_bin_ms  # must match cell 3's factor


# ---- 1. Teacher-forcing predictions ----
cache_model = trained_models['teacher_forcing']['model']
cache_model.eval()

cache_preds_tf, cache_trues = [], []
with torch.no_grad():
    for bx, by in test_loader_cache:
        bx, by_dev = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            pred = torch.exp(cache_model(bx, by_dev, mode='teacher_forcing')).cpu().numpy()
        cache_preds_tf.append(pred)
        cache_trues.append(by.numpy())

cache_pred_tf = np.concatenate(cache_preds_tf, axis=0)
cache_true    = np.concatenate(cache_trues, axis=0)

# ---- 2. Fully-autoregressive predictions ----
print("Computing AR predictions (sequential, may take a moment)...")
cache_model = trained_models['ar']['model']
cache_model.eval()

cache_preds_ar = []
with torch.no_grad():
    for bx, by in test_loader_cache:
        bx = bx.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            pred = torch.exp(cache_model(bx, spike_context=None, mode='ar')).cpu().numpy()
        cache_preds_ar.append(pred)

cache_pred_ar = np.concatenate(cache_preds_ar, axis=0)

# ---- 3. Semi-autoregressive predictions ----
print("Computing semi-AR predictions (sequential, may take a moment)...")
cache_model = trained_models['semi_ar']['model']
cache_model.eval()
cache_preds_semi = []
with torch.no_grad():
    for bx, by in test_loader_cache:
        bx, by_dev = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            pred = torch.exp(cache_model(bx, spike_context=by_dev, mode='semi_ar')).cpu().numpy()
        cache_preds_semi.append(pred)

cache_pred_semi = np.concatenate(cache_preds_semi, axis=0)

# ---- 4. Coarsen all arrays ----
if coarse_factor > 1:
    cache_pred_tf   = _coarsen(cache_pred_tf,   factor=coarse_factor)
    cache_pred_ar   = _coarsen(cache_pred_ar,   factor=coarse_factor)
    cache_pred_semi = _coarsen(cache_pred_semi, factor=coarse_factor)
    cache_true      = _coarsen(cache_true,      factor=coarse_factor)

# ---- 5. Per-neuron metrics ----
n_neurons_here = cache_pred_tf.shape[1]

tf_corrs = np.zeros(n_neurons_here)
ar_cache_corrs = np.zeros(n_neurons_here)
semi_ar_corrs = np.zeros(n_neurons_here)
for n in range(n_neurons_here):
    tt = cache_true[:, n, :].ravel()
    pt = cache_pred_tf[:, n, :].ravel()
    pa = cache_pred_ar[:, n, :].ravel()
    ps = cache_pred_semi[:, n, :].ravel()
    if pt.std() > 1e-8 and tt.std() > 1e-8:
        tf_corrs[n] = pearsonr(pt, tt)[0]
    if pa.std() > 1e-8 and tt.std() > 1e-8:
        ar_cache_corrs[n] = pearsonr(pa, tt)[0]
    if ps.std() > 1e-8 and tt.std() > 1e-8:
        semi_ar_corrs[n] = pearsonr(ps, tt)[0]

tf_fve_neurons, tf_fve_mean = fraction_variance_explained(
    cache_true, cache_pred_tf, global_variance=True)
ar_fve_cache_neurons, ar_fve_cache_mean = fraction_variance_explained(
    cache_true, cache_pred_ar, global_variance=True)
semi_ar_fve_neurons, semi_ar_fve_mean = fraction_variance_explained(
    cache_true, cache_pred_semi, global_variance=True)

print(f"\n=== Cache CNN ({COARSE_BIN_MS}ms bins) ===")
print(f"{'Mode':<20} {'Mean r':>8} {'Med r':>8} {'Mean FVE':>10} {'Med FVE':>10} {'> LOO (r)':>10} {'> LOO (FVE)':>12}")
print("-" * 80)
for tag, corrs, fve in [("Teacher Forcing", tf_corrs, tf_fve_neurons),
                         ("Semi-AR",         semi_ar_corrs, semi_ar_fve_neurons),
                         ("Autoregressive",  ar_cache_corrs, ar_fve_cache_neurons)]:
    print(f"{tag:<20} {corrs.mean():8.4f} {np.median(corrs):8.4f} "
          f"{fve.mean():10.4f} {np.median(fve):10.4f} "
          f"{(corrs > loo_corrs).sum():>4}/{len(loo_corrs):<4} "
          f"{(fve > loo_fve_neurons).sum():>4}/{len(loo_fve_neurons):<4}")

# ---- 6. Register in MODEL_DATA ----
hist_h = history_cfg.get('history', 1)
MODEL_DATA['cache_tf'] = {
    'corrs': tf_corrs, 'fve': tf_fve_neurons,
    'label': f'Cache CNN TF (cache={CACHE_SIZE})',
}
MODEL_DATA['cache_semi_ar'] = {
    'corrs': semi_ar_corrs, 'fve': semi_ar_fve_neurons,
    'label': f'Cache CNN Semi-AR (cache={CACHE_SIZE})',
}
MODEL_DATA['cache_ar'] = {
    'corrs': ar_cache_corrs, 'fve': ar_fve_cache_neurons,
    'label': f'Cache CNN AR (cache={CACHE_SIZE})',
}
MODEL_DATA['hist_gt'] = {
    'corrs': hist_corrs, 'fve': hist_fve_neurons,
    'label': f'Lagged history (h={hist_h})',
}

# ---- 7. Combined 5-model overlay plot ----
models_to_plot = ['cache_tf', 'cache_semi_ar', 'cache_ar', 'nohist', 'hist_gt']
colors = ['tab:blue', 'tab:cyan', 'tab:orange', 'tab:green', 'tab:red']

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
n_total = len(loo_corrs)

for key, c in zip(models_to_plot, colors):
    d = MODEL_DATA[key]
    n_better_corr = (d['corrs'] > loo_corrs).sum()
    n_better_fve  = (d['fve']   > loo_fve_neurons).sum()

    ax[0].scatter(loo_corrs, d['corrs'], color=c, alpha=0.65, s=25,
                  label=f"{d['label']} ({n_better_corr}/{n_total} > LOO)")
    ax[1].scatter(loo_fve_neurons, d['fve'], color=c, alpha=0.65, s=25,
                  label=f"{d['label']} ({n_better_fve}/{n_total} > LOO)")

for a in ax:
    a.plot([0, 1], [0, 1], 'k--', lw=0.8)
    a.legend(fontsize=8)

ax[0].set_xlabel("LOO Temporal Correlation")
ax[0].set_ylabel("Model Temporal Correlation")
ax[0].set_title(f"Temporal Correlation per Neuron ({COARSE_BIN_MS}ms bins)")

ax[1].set_xlabel("LOO Fraction Variance Explained")
ax[1].set_ylabel("Model FVE")
ax[1].set_title(f"Fraction Variance Explained per Neuron ({COARSE_BIN_MS}ms bins)")

fig.suptitle("Cache CNN (TF vs Semi-AR vs AR) vs Lagged History vs No-History", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Quick diagnostic: are AR predictions exploding?
print("=== Prediction magnitude diagnostics ===")
print(f"{'':20s}  {'mean':>10s}  {'median':>10s}  {'max':>10s}  {'min':>10s}")
print("-" * 65)
for label, arr in [("Ground truth", cache_true),
                   ("TF predictions", cache_pred_tf),
                   ("AR predictions", cache_pred_ar)]:
    print(f"{label:20s}  {arr.mean():10.4f}  {np.median(arr):10.4f}  {arr.max():10.4f}  {arr.min():10.4f}")

print(f"\nAR predictions > 10x GT max: {(cache_pred_ar > 10 * cache_true.max()).sum()} / {cache_pred_ar.size}")
print(f"AR predictions contain NaN:  {np.isnan(cache_pred_ar).sum()}")
print(f"AR predictions contain Inf:  {np.isinf(cache_pred_ar).sum()}")

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy

# ============================================================
# Most-Exciting Input (MEI) via gradient ascent on frozen model
# ============================================================

def find_optimal_stimulus(
    model,
    neuron_indices=None,
    n_stim_channels=42,
    n_input_bins=60,
    stim_range=(-3.0, 3.0),
    lr=0.005,
    norm="l1",
    max_iter=5000,
    convergence_tol=1e-4,
    patience=50,
    reg_penalty=0.01,
    device=None,
    objective='sum',
    seed=42,
    context_fill=None,
    max_rate=None,
):
    """Find the input pattern that maximally drives each neuron.

    Parameters
    ----------
    context_fill : np.ndarray (n_context_channels,) or None
        Fill values for frozen spike-history channels.
    max_rate : float or None
        Clamp predicted rates to <= max_rate (spk/bin) in log-space.

    Returns
    -------
    results : dict with keys:
        'optimal_inputs'       (n_opt, n_stim_channels, n_input_bins)
        'predicted_rates'      (n_opt, n_output_bins)   — target neuron
        'all_predicted_rates'  (n_opt, total_neurons, n_output_bins)  — all neurons
        'objectives'           (n_opt,)
        'converged'            (n_opt,)
        'n_iters'              (n_opt,)
        'neuron_indices'       (n_opt,)
        'n_output_bins'        int
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = deepcopy(model).to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)

    # Detect init_state (valid conv) prepend requirement
    n_prepend = 0
    if hasattr(model, 'use_init_state') and model.use_init_state:
        n_prepend = model.total_conv_reduction
        print(f"  [MEI] init_state valid conv: prepending {n_prepend} bins → output={n_input_bins} bins")
    n_total_bins = n_input_bins + n_prepend

    # Auto-detect model input channels
    n_model_channels = n_stim_channels
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            n_model_channels = m.in_channels
            break
    n_context_channels = n_model_channels - n_stim_channels
    if n_context_channels < 0:
        raise ValueError(
            f"Model expects {n_model_channels} channels but n_stim_channels={n_stim_channels}")

    # Frozen context-channel template (spike history), full temporal extent
    if n_context_channels > 0:
        if context_fill is not None:
            ctx_vals = torch.tensor(context_fill, dtype=torch.float32, device=device)
            if ctx_vals.numel() != n_context_channels:
                raise ValueError(
                    f"context_fill has {ctx_vals.numel()} values but need {n_context_channels}")
            ctx_template = ctx_vals.reshape(1, n_context_channels, 1).expand(
                1, n_context_channels, n_total_bins).contiguous()
            fill_desc = f"mean rates [{ctx_vals.min():.4f}, {ctx_vals.max():.4f}]"
        else:
            ctx_template = torch.zeros(1, n_context_channels, n_total_bins, device=device)
            fill_desc = "zeros"
        print(f"  [MEI] {n_model_channels} total channels; "
              f"optimising {n_stim_channels} stim, padding {n_context_channels} history with {fill_desc}")

    if n_prepend > 0:
        stim_prepend = torch.zeros(1, n_stim_channels, n_prepend, device=device)

    log_max_rate = None
    if max_rate is not None:
        log_max_rate = float(np.log(max_rate))
        print(f"  [MEI] Rate cap: ≤{max_rate} spk/bin (log ≤ {log_max_rate:.4f})")

    def _build_model_input(stim_tensor):
        if n_prepend > 0:
            full_stim = torch.cat(
                [stim_prepend.expand(stim_tensor.shape[0], -1, -1), stim_tensor], dim=2)
        else:
            full_stim = stim_tensor
        if n_context_channels == 0:
            return full_stim
        return torch.cat(
            [full_stim, ctx_template.expand(stim_tensor.shape[0], -1, -1)], dim=1)

    # Verify output shape
    with torch.no_grad():
        dummy_out = model(_build_model_input(
            torch.zeros(1, n_stim_channels, n_input_bins, device=device)))
    total_neurons = dummy_out.shape[1]
    n_output_bins = dummy_out.shape[2]
    print(f"  [MEI] Verified: input ({n_model_channels}, {n_total_bins}) "
          f"→ output ({total_neurons} neurons, {n_output_bins} bins)")
    assert n_output_bins == n_input_bins, (
        f"Expected {n_input_bins} output bins, got {n_output_bins}")

    if neuron_indices is None:
        neuron_indices = list(range(total_neurons))
    n_opt = len(neuron_indices)

    # Storage
    optimal_inputs       = np.zeros((n_opt, n_stim_channels, n_input_bins), dtype=np.float32)
    predicted_rates      = np.zeros((n_opt, n_output_bins), dtype=np.float32)
    all_predicted_rates  = np.zeros((n_opt, total_neurons, n_output_bins), dtype=np.float32)
    objectives           = np.zeros(n_opt, dtype=np.float32)
    converged            = np.zeros(n_opt, dtype=bool)
    n_iters_arr          = np.zeros(n_opt, dtype=int)

    torch.manual_seed(seed)

    for i, nidx in enumerate(neuron_indices):
        stim = (torch.empty(1, n_stim_channels, n_input_bins, device=device)
                .uniform_(stim_range[0], stim_range[1]))
        stim.requires_grad_(True)
        optimizer = torch.optim.Adam([stim], lr=lr)

        best_obj = -float('inf')
        plateau_count = 0

        for it in range(max_iter):
            optimizer.zero_grad()

            log_rates = model(_build_model_input(stim))  # (1, n_neurons, n_output_bins)
            if log_max_rate is not None:
                log_rates = torch.clamp(log_rates, max=log_max_rate)

            neuron_log_rates = log_rates[0, nidx, :]
            if objective == 'sum':
                obj = neuron_log_rates.sum()
            elif objective == 'peak':
                obj = torch.logsumexp(neuron_log_rates, dim=0)
            else:
                raise ValueError(f"Unknown objective '{objective}'")

            if norm == "l1":
                stim_norm = stim.abs().sum()
            elif norm == "l2":
                stim_norm = (stim ** 2).sum()
            else:
                stim_norm = 0.5 * (stim ** 2).sum() + 0.5 * stim.abs().sum()

            loss = -(obj - reg_penalty * stim_norm)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                stim.clamp_(*stim_range)

            cur_obj = obj.item()
            if abs(cur_obj - best_obj) < convergence_tol:
                plateau_count += 1
            else:
                plateau_count = 0
            if cur_obj > best_obj:
                best_obj = cur_obj
            if plateau_count >= patience:
                converged[i] = True
                n_iters_arr[i] = it + 1
                break
        else:
            n_iters_arr[i] = max_iter

        # Store target + all-neuron rates from the final stimulus
        with torch.no_grad():
            final_all_log = model(_build_model_input(stim))[0]  # (total_neurons, n_output_bins)
            if log_max_rate is not None:
                final_all_log = torch.clamp(final_all_log, max=log_max_rate)
            final_all_rates = torch.exp(final_all_log).cpu().numpy()

        optimal_inputs[i]      = stim.detach().cpu().numpy()[0]
        predicted_rates[i]     = final_all_rates[nidx]
        all_predicted_rates[i] = final_all_rates
        objectives[i]          = best_obj

        if (i + 1) % 10 == 0 or i == 0:
            status = "converged" if converged[i] else f"{n_iters_arr[i]} iters"
            print(f"  Neuron {nidx:3d} — obj={best_obj:.4f}  "
                  f"rate: mean={final_all_rates[nidx].mean():.4f} "
                  f"max={final_all_rates[nidx].max():.4f}  ({status})")

    print(f"\nDone: {converged.sum()}/{n_opt} neurons converged")
    return {
        'optimal_inputs':      optimal_inputs,
        'predicted_rates':     predicted_rates,      # target neuron
        'all_predicted_rates': all_predicted_rates,  # all neurons
        'objectives':          objectives,
        'converged':           converged,
        'n_iters':             n_iters_arr,
        'neuron_indices':      np.array(neuron_indices),
        'n_output_bins':       n_output_bins,
    }


def plot_mei_results(results, top_k=5, save_dir=None, mean_rates=None):
    """Visualise MEI results.

    Right panel: target neuron's rate (black) + all other neurons' rates
    for the same stimulus (faint blue), to show stimulus selectivity.
    Units: spikes per 10ms bin.
    """
    n_opt = len(results['neuron_indices'])
    nidx  = results['neuron_indices']
    objs  = results['objectives']
    n_bins = results['predicted_rates'].shape[1]
    mei_total_spikes = results['predicted_rates'].sum(axis=1)

    # ---- 1. Bar chart: total spikes (same units as mean-rate baseline) ----
    fig, ax = plt.subplots(figsize=(max(8, n_opt * 0.2), 4))
    colors = ['tab:green' if c else 'tab:orange' for c in results['converged']]
    ax.bar(range(n_opt), mei_total_spikes, color=colors, alpha=0.8,
           label='MEI total predicted spikes')
    if mean_rates is not None:
        mean_sums = mean_rates[nidx] * n_bins
        ax.bar(range(n_opt), mean_sums, color='red', alpha=0.55,
               edgecolor='darkred', linewidth=0.8,
               label=f'Mean rate × {n_bins} bins (baseline)')
        ax.legend(fontsize=8, loc='upper right')
    else:
        ax.legend(fontsize=8)
    ax.set_xticks(range(n_opt))
    ax.set_xticklabels(nidx, fontsize=6, rotation=90)
    ax.set_xlabel('Neuron index')
    ax.set_ylabel('Total expected spikes per trial\n(sum over 60 bins × 10ms)')
    ax.set_title('MEI: total predicted spikes per neuron')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, 'mei_objectives.png'), dpi=200, bbox_inches='tight')
    plt.show()

    # ---- 2 & 3. Top-k: stimulus heatmap + selectivity rate traces ----
    ranked = np.argsort(objs)[::-1][:top_k]
    fig, axes = plt.subplots(top_k, 2, figsize=(14, 3 * top_k))
    if top_k == 1:
        axes = axes[np.newaxis, :]

    has_all = 'all_predicted_rates' in results

    for row, idx in enumerate(ranked):
        ni   = nidx[idx]
        inp  = results['optimal_inputs'][idx]
        rate = results['predicted_rates'][idx]

        # --- left: stimulus heatmap ---
        im = axes[row, 0].imshow(inp, aspect='auto', cmap='RdBu_r',
                                  vmin=-3, vmax=3, interpolation='nearest')
        axes[row, 0].set_ylabel(f'Neuron {ni}\n(obj={objs[idx]:.2f})', fontsize=9)
        axes[row, 0].set_xlabel('Time bin (10ms)')
        if row == 0:
            axes[row, 0].set_title('Optimal stimulus (MEI)')
        plt.colorbar(im, ax=axes[row, 0], shrink=0.7, label='Current (µA)')

        # --- right: rate traces ---
        ax_r = axes[row, 1]

        # Other neurons in faint blue
        if has_all:
            all_rates = results['all_predicted_rates'][idx]  # (total_neurons, n_output_bins)
            n_total = all_rates.shape[0]
            for j in range(n_total):
                if j != ni:
                    ax_r.plot(all_rates[j], color='steelblue', lw=0.6, alpha=0.15)
            ax_r.plot([], [], color='steelblue', lw=1.5, alpha=0.5,
                      label=f'Other neurons (n={n_total - 1})')

        # Target neuron on top
        ax_r.plot(rate, 'k-', lw=2.0, label=f'Target neuron {ni}', zorder=5)
        ax_r.fill_between(range(len(rate)), rate, alpha=0.2, color='k', zorder=4)

        # Mean-rate reference line
        if mean_rates is not None:
            mr = mean_rates[ni]
            ax_r.axhline(mr, color='tab:red', ls='--', lw=1.2,
                         label=f'Mean data rate ({mr:.3f} spk/bin)')
            gain = rate.mean() / mr if mr > 1e-8 else float('inf')
            ax_r.text(0.98, 0.95, f'{gain:.1f}× mean',
                      transform=ax_r.transAxes, ha='right', va='top', fontsize=8,
                      bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

        ax_r.set_xlabel('Time bin (10ms)')
        ax_r.set_ylabel('Predicted rate (spk/bin)')
        if row == 0:
            ax_r.set_title('Predicted rates — target (black) vs all neurons (blue)')
        ax_r.legend(fontsize=7, loc='upper left')
        ax_r.grid(alpha=0.3)

    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, 'mei_top_patterns.png'), dpi=200, bbox_inches='tight')
    plt.show()
    return fig


print("MEI functions defined: find_optimal_stimulus(), plot_mei_results()")

In [ ]:
norm = "l1"
reg_penalty = 0.1
convergence_tol = 1e-3
max_iter = 50000
lr = 5e-4
mei_objective = "sum"

# ---- Compute mean firing rate per neuron from test data (for reference) ----
# Use the underlying dataset directly to avoid DataLoader shared-memory issues on macOS
_all_targets = []
for _i in range(len(nonhistory_test_loader.dataset)):
    _, _by = nonhistory_test_loader.dataset[_i]
    _all_targets.append(_by.numpy())
_all_targets = np.stack(_all_targets, axis=0)               # (n_trials, n_neurons, T)
mean_firing_rates = _all_targets.mean(axis=(0, 2))          # (n_neurons,)
print(f"Mean firing rates (spk/bin): min={mean_firing_rates.min():.4f}, max={mean_firing_rates.max():.4f}, "
      f"median={np.median(mean_firing_rates):.4f}")

# ---- Run MEI on the no-history baseline CNN ----
print(f"\nModel: No-history CNN from {nonhistory_cfg.get('model_type', 'cnn')}")
print(f"Input shape: ({n_stim_channels}, 60)  |  Output neurons: {n_neurons}")

mei_results = find_optimal_stimulus(
    model=nonhistory_model,
    neuron_indices=None,         # all 63 neurons
    n_stim_channels=n_stim_channels,
    n_input_bins=60,
    stim_range=(-3.0, 3.0),
    lr=lr,
    max_iter=max_iter,
    convergence_tol=convergence_tol,
    patience=50,
    norm=norm,
    reg_penalty=reg_penalty,
    objective=mei_objective,
    seed=42,
    max_rate=1.0,                # ≤1 spike per 10ms bin
)

# ---- Visualise ----
SAVE_DIR = os.path.join(SWEEP_DIR, 'mei_nohist')
plot_mei_results(mei_results, top_k=8, save_dir=SAVE_DIR, mean_rates=mean_firing_rates)

# ---- Save results ----
np.savez(os.path.join(SAVE_DIR, 'mei_results.npz'), **mei_results)
print(f"Saved to {SAVE_DIR}")

In [ ]:
# ---- Run MEI on the history CNN ----
# The history model expects 42 stim + 63 spike-history channels.
# We optimise only the stim channels; history channels are filled with
# the per-neuron mean firing rate from the test set.
print(f"Model: History CNN")
print(f"Input shape: ({n_stim_channels} stim + {n_neurons} history, 60)  |  Output neurons: {n_neurons}")

mei_results = find_optimal_stimulus(
    model=history_model,
    neuron_indices=None,         # all 63 neurons
    n_stim_channels=n_stim_channels,
    n_input_bins=60,
    stim_range=(-3.0, 3.0),
    lr=lr,
    max_iter=max_iter,
    convergence_tol=convergence_tol,
    patience=50,
    norm=norm,
    reg_penalty=reg_penalty,
    objective=mei_objective,
    seed=42,
    context_fill=mean_firing_rates,   # pad history channels with mean rates
    max_rate=1.0,                     # ≤1 spike per 10ms bin
)

# ---- Visualise ----
SAVE_DIR = os.path.join(SWEEP_DIR, 'mei_history')
plot_mei_results(mei_results, top_k=3, save_dir=SAVE_DIR, mean_rates=mean_firing_rates)

# ---- Save results ----
np.savez(os.path.join(SAVE_DIR, 'mei_results.npz'), **mei_results)
print(f"Saved to {SAVE_DIR}")

In [ ]:
# Diagnostic: check both models
print(f"nonhistory: init_state={nonhistory_model.use_init_state}, kernel_sizes={nonhistory_model.kernel_sizes_list}, reduction={nonhistory_model.total_conv_reduction}")
print(f"history:    init_state={history_model.use_init_state}, kernel_sizes={history_model.kernel_sizes_list}, reduction={history_model.total_conv_reduction}")
print(f"\nnonhistory_cfg kernel_sizes: {nonhistory_cfg.get('kernel_sizes')}")
print(f"nonhistory_cfg init_state: {nonhistory_cfg.get('init_state')}")

# Check actual output shapes
import torch
dummy_nh = torch.zeros(1, 42, 60)
with torch.no_grad():
    out_nh = nonhistory_model(dummy_nh.to(next(nonhistory_model.parameters()).device))
print(f"\nNonhistory: input (1, 42, 60) → output {out_nh.shape}")

dummy_nh2 = torch.zeros(1, 42, 60 + nonhistory_model.total_conv_reduction)
with torch.no_grad():
    out_nh2 = nonhistory_model(dummy_nh2.to(next(nonhistory_model.parameters()).device))
print(f"Nonhistory: input (1, 42, {60 + nonhistory_model.total_conv_reduction}) → output {out_nh2.shape}")

# Check nonhistory mei_results shape
print(f"\nmei_results predicted_rates shape: {mei_results['predicted_rates'].shape}")
print(f"mei_results optimal_inputs shape: {mei_results['optimal_inputs'].shape}")